# Exercise 1 — From Pixels to Visual Evidence

**Computer Vision for Industrial Systems**

This first exercise is about non-ML computer vision fundamentals.  
You will work with sampling, aliasing, blur, noise, saturation, gamma, thresholding, morphology, and simple measurement.

## Learning objectives

After this notebook, you should be able to:

1. compute object-space resolution in **mm/px**,
2. reason about whether a feature is sufficiently sampled,
3. demonstrate aliasing by downsampling,
4. simulate blur, noise, saturation, bit depth, and gamma effects,
5. create a simple thresholding + morphology + measurement pipeline,
6. write a short engineering interpretation before using any machine-learning model.

## How to work with this notebook

You will see code cells with TODO blocks, for example:

```python
# TODO: implement this function
...
```

Replace the `...` with your code and run the cell.

The tasks have three levels:

- 🟢 **Basic** — should be solved by everyone during the exercise
- 🟡 **Intermediate** — slightly more reasoning / implementation
- 🔴 **Advanced** — optional challenge if you finish early

In [ ]:
from pathlib import Path
import sys
import math
import os
# Ensure Matplotlib uses a writable cache directory in Binder/sandbox environments.
os.environ.setdefault("MPLCONFIGDIR", str((Path.cwd() / ".matplotlib_cache").resolve()))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Make helper module importable when running from notebooks/
repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(repo_root / "src"))

from cvis_utils import (
    load_gray, show_images, plot_histograms,
    mm_per_pixel, pixels_per_feature,
    downsample_image, motion_blur_kernel, apply_motion_blur,
    add_gaussian_noise, gamma_correct, quantize_image,
    sobel_energy, laplacian_variance,
    otsu_segment, clean_mask, region_table, iou
)

DATA = repo_root / "data"
SYN = DATA / "synthetic"
IND = DATA / "industrial_like"
print("Repository root:", repo_root)
print("Synthetic data exists:", SYN.exists())
print("Industrial-like data exists:", IND.exists())

## 0. Load and inspect images

We use two types of data:

1. **synthetic images** for controlled sampling/aliasing demonstrations,
2. **industrial-style generated surface images** for defect visibility, segmentation, and measurement.

These industrial-style images are generated for teaching. They are not a substitute for real benchmark datasets such as KolektorSDD or MVTec AD.

In [ ]:
checker = load_gray(SYN / "checkerboard_highres.png")
brick = load_gray(SYN / "brick_wall_synthetic.png")
surface = load_gray(IND / "images" / "surface_defect_bright_scratch_easy.png")
mask_gt = load_gray(IND / "masks" / "surface_defect_bright_scratch_easy_mask.png") > 0.5

show_images(
    [checker, brick, surface, mask_gt],
    ["checkerboard", "brick-like texture", "surface defect", "ground-truth mask"],
    ncols=4,
    figsize=(14, 4),
)

# Part A — Object-space resolution, sampling, and aliasing

Industrial question:

> Is the relevant feature large enough in pixel space to be detected or measured reliably?

We use the running example from the lecture:

- field of view width: **400 mm**
- horizontal pixel count: **2448 px**
- defect size: **0.5 mm**

## Task A1 — Compute mm/px and pixels per defect 🟢

Implement the two functions below.

Hints:
- object-space resolution:  
  `mm_per_px = field_of_view_mm / number_of_pixels`
- pixels per feature:  
  `pixels = feature_size_mm / mm_per_px`

In [ ]:
def student_mm_per_pixel(fov_mm: float, n_pixels: int) -> float:
    # TODO: return the object-space resolution in mm/px
    ...


def student_pixels_per_feature(feature_size_mm: float, mm_per_px: float) -> float:
    # TODO: return how many pixels span the feature
    ...

In [ ]:
fov_mm = 400.0
n_px = 2448
defect_size_mm = 0.5

mm_px = student_mm_per_pixel(fov_mm, n_px)
px_defect = student_pixels_per_feature(defect_size_mm, mm_px)

print(f"Object-space resolution: {mm_px:.4f} mm/px")
print(f"0.5 mm defect spans: {px_defect:.2f} px")

### Short interpretation A1 🟢

Write 2–3 sentences:

1. Is the 0.5 mm defect sufficiently sampled?
2. Would you trust this setup for classification? For localization? For measurement?
3. What additional image-quality factors matter besides pixel count?

In [ ]:
# TODO: write your short interpretation as a string.
answer_A1 = """


"""
print(answer_A1)

## Task A2 — Downsampling and aliasing 🟢 / 🟡

Downsample the brick-like image to a smaller size with and without anti-aliasing.

Your goal is to visually compare:
- downsampling **without** anti-aliasing,
- downsampling **with** anti-aliasing.

This illustrates why fine structures can create false patterns when sampled too coarsely.

In [ ]:
scale = 0.18  # try 0.30, 0.18, 0.10

# TODO: create two downsampled versions of `brick`
# one with anti_aliasing=False
# one with anti_aliasing=True
brick_bad = ...
brick_good = ...

show_images(
    [brick, brick_bad, brick_good],
    ["original", "downsampled, no anti-aliasing", "downsampled, anti-aliasing"],
    ncols=3,
    figsize=(13, 4),
)

## Task A3 — Intensity profiles 🟡

Choose one image row and plot the intensity profile of the original and downsampled images.

This helps you see how fine structure is transformed by sampling.

In [ ]:
row_original = brick.shape[0] // 2
row_bad = brick_bad.shape[0] // 2
row_good = brick_good.shape[0] // 2

plt.figure(figsize=(12, 4))
# TODO: plot three line profiles:
# original brick row, bad downsampled row, good downsampled row
# HINT: use plt.plot(...)
...
plt.title("Intensity profiles")
plt.xlabel("column index")
plt.ylabel("intensity")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

# Part B — Blur, noise, saturation, bit depth, and gamma

Industrial question:

> How do imaging perturbations change the evidence available to a downstream algorithm?

We simulate common effects from the lecture:
- motion blur,
- noise,
- saturation/clipping,
- reduced bit depth,
- gamma/display processing.

## Task B1 — Motion blur and edge energy 🟢

Apply motion blur to the surface-defect image and compare sharpness / edge evidence.

In [ ]:
# TODO: apply motion blur to the `surface` image
# HINT: use apply_motion_blur(surface, length=..., angle_deg=...)
surface_blur = ...

show_images(
    [surface, surface_blur],
    ["original surface defect", "motion-blurred"],
    ncols=2,
    figsize=(10, 4),
)

print("Sobel energy original:", sobel_energy(surface))
print("Sobel energy blurred :", sobel_energy(surface_blur))
print("Laplacian variance original:", laplacian_variance(surface))
print("Laplacian variance blurred :", laplacian_variance(surface_blur))

## Task B2 — Noise, saturation, and bit depth 🟢 / 🟡

Create three modified versions of the image:

1. noisy image,
2. saturated / clipped image,
3. quantized image with low bit depth.

Then plot histograms.

In [ ]:
# TODO: create noisy, clipped, and quantized versions
surface_noisy = ...
surface_clipped = ...
surface_quantized = ...

show_images(
    [surface, surface_noisy, surface_clipped, surface_quantized],
    ["original", "noisy", "clipped / saturated", "quantized"],
    ncols=4,
    figsize=(14, 4),
)

plot_histograms(
    [surface, surface_noisy, surface_clipped, surface_quantized],
    ["original", "noisy", "clipped", "quantized"],
    bins=64,
    figsize=(14, 3),
)

## Task B3 — Gamma correction and measurement meaning 🟡

Apply gamma correction and compare the histogram.

Remember from the lecture:

> Display processing can hide measurement assumptions.

Gamma correction can be useful for visualization but changes the relationship between pixel intensity and physical signal.

In [ ]:
gamma = 2.2

# TODO: apply gamma correction
surface_gamma = ...

show_images([surface, surface_gamma], ["original / linear-like", f"gamma corrected, gamma={gamma}"], ncols=2, figsize=(10, 4))
plot_histograms([surface, surface_gamma], ["original", "gamma corrected"], bins=64, figsize=(10, 3))

### Short interpretation B 🟢

Write 4–6 sentences:

1. Which perturbation changed the image most strongly?
2. Which perturbation made the defect hardest to see?
3. Did a visually “nicer” image always preserve measurement meaning?
4. What would you change in the imaging setup before trying a model?

In [ ]:
answer_B = """


"""
print(answer_B)

# Part C — Thresholding, morphology, and simple measurement

Industrial question:

> Can a simple non-ML pipeline isolate and measure the defect?

We will implement:

1. grayscale thresholding,
2. morphological cleanup,
3. connected-component measurement,
4. optional mask quality using IoU.

## Task C1 — Otsu thresholding 🟢

Segment the scratch using Otsu's threshold.

For this specific scratch image, the defect is brighter than the background.

In [ ]:
# TODO: segment the bright scratch with Otsu threshold
mask_raw, threshold_value = ...

print("Otsu threshold:", threshold_value)
show_images([surface, mask_raw], ["surface image", "raw threshold mask"], ncols=2, figsize=(10, 4))

## Task C2 — Morphological cleanup 🟢

Remove small objects and close small gaps in the mask.

In [ ]:
# TODO: clean the raw mask
mask_clean = ...

show_images([mask_raw, mask_clean, mask_gt], ["raw mask", "cleaned mask", "ground truth mask"], ncols=3, figsize=(12, 4))
print("IoU raw mask   :", iou(mask_raw, mask_gt))
print("IoU clean mask :", iou(mask_clean, mask_gt))

## Task C3 — Region measurement 🟢 / 🟡

Measure connected components in the cleaned mask.

Use your `mm_px` from Part A to convert area from pixels² to mm².

In [ ]:
# TODO: compute region table using mask_clean
regions = ...

df_regions = pd.DataFrame(regions)
df_regions

## Task C4 — Try a harder image 🔴

Repeat thresholding and measurement for a harder defect image.

Options:
- `surface_defect_dark_scratch_01.png` — dark scratch instead of bright scratch
- `surface_defect_spot_01.png` — compact spot defect
- `surface_defect_edge_chip_01.png` — defect at the image boundary

Think carefully:
- Is the defect brighter or darker than the background?
- Does the same thresholding strategy still work?
- Does morphology help or hurt?

In [ ]:
hard_image_path = IND / "images" / "surface_defect_dark_scratch_01.png"
hard_mask_path = IND / "masks" / "surface_defect_dark_scratch_01_mask.png"

hard_img = load_gray(hard_image_path)
hard_gt = load_gray(hard_mask_path) > 0.5

# TODO: choose foreground="bright" or foreground="dark"
hard_raw, hard_thr = ...
hard_clean = ...

show_images([hard_img, hard_raw, hard_clean, hard_gt], ["hard image", "raw mask", "clean mask", "ground truth"], ncols=4, figsize=(14, 4))
print("Hard image threshold:", hard_thr)
print("Hard image IoU:", iou(hard_clean, hard_gt))

# Final engineering interpretation

Write a short engineering recommendation.

Imagine you are the engineer responsible for this inspection setup.  
Before using a machine-learning model, what would you check or improve?

In [ ]:
final_recommendation = """
1. Sampling / resolution:


2. Blur / noise / saturation / gamma:


3. Thresholding / segmentation:


4. Imaging setup recommendation before ML:


"""
print(final_recommendation)

## Optional extension ideas

If you finish early:

1. Try different motion-blur lengths and plot sharpness metrics.
2. Build a small table comparing IoU for multiple defects.
3. Try adaptive thresholding or edge detection.
4. Create your own synthetic defect and test whether the pipeline can detect it.
5. Write a function that estimates the maximum exposure time for a given blur tolerance.